In [1]:
import ROOT

mass = ROOT.RooRealVar("mass", "Massa do B", 5.0, 5.6, "GeV")
pt   = ROOT.RooRealVar("pt", "pT do B", 0, 50, "GeV")
observables = ROOT.RooArgSet(mass, pt)

# Histogramas para preencher manualmente
hist_pt_data = ROOT.TH1F("hist_pt_data", "pT Dados Reais", 50, 0, 50)
hist_pt_mc   = ROOT.TH1F("hist_pt_mc", "pT Monte Carlo", 50, 0, 50)

# Dataset do RooFit que será usado para o Fit e sPlot
data_roofit = ROOT.RooDataSet("data", "Dataset para Fit", observables)

# ==============================================================================
# 2. LENDO DADOS REAIS (DATA)
# ==============================================================================
print(">>> Lendo arquivo de DADOS...")

file_data = ROOT.TFile("/home/tandrade/Flavour_Anomalies/Data/Data_Collision.root")
tree_data = file_data.Get("tree_ML")

# Otimização: Desliga ramos que não vamos usar para ler mais rápido
tree_data.SetBranchStatus("*", 0)
tree_data.SetBranchStatus("BToTrkTrkMuMu_fit_mass_Kpi_good", 1)
tree_data.SetBranchStatus("BToTrkTrkMuMu_fit_pt_good", 1)
tree_data.SetBranchStatus("MuMu_mass_idx_good", 1)

count_data = 0
for event in tree_data:
    masses = event.BToTrkTrkMuMu_fit_mass_Kpi_good
    pts = event.BToTrkTrkMuMu_fit_pt_good
    mumu_masses = event.MuMu_mass_idx_good  
    
    if len(masses) == 0: continue

    # Loop sobre os candidatos dentro do evento
    for mass_val, pt_val, mumu_val in zip(masses, pts, mumu_masses):        
        
        # APLICANDO CORTES
        # 1. Janela de massa do J/Psi
        if mumu_val > 2.9 and mumu_val < 3.3: 
            # 2. Janela de massa do B (nosso sinal)
            if mass_val > 5.0 and mass_val < 5.6:
                
                # Preenche histograma normal
                hist_pt_data.Fill(pt_val)
                
                # Preenche dataset do RooFit
                mass.setVal(mass_val)
                pt.setVal(pt_val)
                data_roofit.add(observables)
                count_data += 1

print(f"   -> Eventos selecionados nos Dados: {count_data}")

# ==============================================================================
# 3. LENDO MONTE CARLO (MC)
# ==============================================================================
print(">>> Lendo arquivo de MC...")

file_mc = ROOT.TFile("/home/tandrade/Flavour_Anomalies/Data/MC_JPsiKstar.root")
tree_mc = file_mc.Get("tree_ML")

tree_mc.SetBranchStatus("*", 0)
tree_mc.SetBranchStatus("BToTrkTrkMuMu_fit_mass_Kpi_good", 1)
tree_mc.SetBranchStatus("BToTrkTrkMuMu_fit_pt_good", 1)
tree_mc.SetBranchStatus("MuMu_mass_idx_good", 1)

count_mc = 0
for event in tree_mc:
    masses = event.BToTrkTrkMuMu_fit_mass_Kpi_good
    pts = event.BToTrkTrkMuMu_fit_pt_good
    mumu_masses = event.MuMu_mass_idx_good  
    
    if len(masses) == 0: continue

    for mass_val, pt_val, mumu_val in zip(masses, pts, mumu_masses):        
        # MESMOS CORTES DOS DADOS
        if mumu_val > 2.9 and mumu_val < 3.3:
            if mass_val > 5.0 and mass_val < 5.6:
                hist_pt_mc.Fill(pt_val)
                count_mc += 1

print(f"   -> Eventos selecionados no MC: {count_mc}")

# Garante que os histogramas não sumam se fecharmos o arquivo (opcional aqui, mas boa prática)
hist_pt_mc.SetDirectory(0)
hist_pt_data.SetDirectory(0)

# ==============================================================================
# 4. CONSTRUINDO O MODELO (PDFs)
# ==============================================================================
print(">>> Construindo modelo de Fit...")

# --- SINAL (Crystal Ball) ---
mean  = ROOT.RooRealVar("mean", "Média", 5.28, 5.26, 5.30)
sigma = ROOT.RooRealVar("sigma", "Largura", 0.025, 0.01, 0.05)
alpha = ROOT.RooRealVar("alpha", "Alpha (Cauda)", 1.5, 0.5, 3.0)
n     = ROOT.RooRealVar("n", "Expoente n", 5.0, 1.0, 15.0)
cb_pdf = ROOT.RooCBShape("cb_pdf", "Crystal Ball Signal", mass, mean, sigma, alpha, n)

# --- BACKGROUND (Exponencial) ---
tau = ROOT.RooRealVar("tau", "Decaimento Exp", -2.0, -10.0, 0.0)
exp_pdf = ROOT.RooExponential("exp_pdf", "Background Exp", mass, tau)

# --- MODELO TOTAL ---
# Números esperados de eventos (inicialização)
n_entries = data_roofit.numEntries()
nsig = ROOT.RooRealVar("n_sig", "N Eventos Sinal", n_entries * 0.4, 0, n_entries)
nbkg = ROOT.RooRealVar("n_bkg", "N Eventos Fundo", n_entries * 0.6, 0, n_entries)

model = ROOT.RooAddPdf("model", "Sinal + Fundo", 
                       ROOT.RooArgList(cb_pdf, exp_pdf), 
                       ROOT.RooArgList(nsig, nbkg))

# ==============================================================================
# 5. REALIZANDO O FIT
# ==============================================================================
print(">>> Rodando fit...")
fit_result = model.fitTo(data_roofit, 
                         ROOT.RooFit.Save(True), 
                         ROOT.RooFit.PrintLevel(-1), 
                         ROOT.RooFit.Extended(True))

# Salva o plot do fit para conferência
c_fit = ROOT.TCanvas("c_fit", "Resultado do Fit", 800, 600)
frame_mass = mass.frame()
data_roofit.plotOn(frame_mass)
model.plotOn(frame_mass)
model.plotOn(frame_mass, ROOT.RooFit.Components("cb_pdf"), ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.LineStyle(ROOT.kDashed))
model.plotOn(frame_mass, ROOT.RooFit.Components("exp_pdf"), ROOT.RooFit.LineColor(ROOT.kGreen), ROOT.RooFit.LineStyle(ROOT.kDashed))
frame_mass.Draw()
c_fit.SaveAs("check_fit_mass.png")

# ==============================================================================
# 6. CÁLCULO DO sPLOT (UNFOLDING)
# ==============================================================================
print(">>> Calculando sWeights...")

# Importante: Fixar os parâmetros de forma para o sPlot
mean.setConstant(True)
sigma.setConstant(True)
alpha.setConstant(True)
n.setConstant(True)
tau.setConstant(True)

# Calcula os pesos
splot = ROOT.RooStats.SPlot("sData", "sPlot Calculation", data_roofit, model, ROOT.RooArgList(nsig, nbkg))

print(">>> Extraindo histogramas ponderados...")

# Histogramas para guardar o resultado do sPlot
hist_splot_sig = ROOT.TH1F("hist_splot_sig", "Sinal Extraído (sPlot)", 50, 0, 50)
hist_splot_bkg = ROOT.TH1F("hist_splot_bkg", "Fundo Extraído (sPlot)", 50, 0, 50)

# Iterar sobre o dataset para pegar os pesos e preencher os histogramas
for i in range(data_roofit.numEntries()):
    row = data_roofit.get(i)
    
    # Valor da variável de controle (pT)
    val_pt = row.getRealValue("pt")
    
    # Pesos calculados pelo sPlot (nomes são padrão: nome_var + _sw)
    weight_sig = row.getRealValue("n_sig_sw")
    weight_bkg = row.getRealValue("n_bkg_sw")
    
    hist_splot_sig.Fill(val_pt, weight_sig)
    hist_splot_bkg.Fill(val_pt, weight_bkg)

# ==============================================================================
# 7. COMPARAÇÃO FINAL: sPLOT (DADOS) vs MC
# ==============================================================================
print(">>> Gerando plot de comparação final...")

c_comp = ROOT.TCanvas("c_comp", "Comparacao Final", 800, 600)

# Estética - Dados (sPlot)
hist_splot_sig.SetLineColor(ROOT.kBlack)
hist_splot_sig.SetMarkerStyle(20) # Bolinhas
hist_splot_sig.SetMarkerSize(0.8)
hist_splot_sig.SetTitle("Comparacao: Dados (sPlot) vs Monte Carlo;pT [GeV];Eventos (Normalizado)")

# Estética - MC
hist_pt_mc.SetLineColor(ROOT.kRed)
hist_pt_mc.SetLineWidth(2)
hist_pt_mc.SetFillColorAlpha(ROOT.kRed, 0.3)

# NORMALIZAÇÃO:
# Precisamos escalar o MC para ter a mesma área que os dados sPlot
# Se não fizermos isso, um terá 1 milhão de eventos e o outro 10 mil, impossível comparar a forma.
if hist_pt_mc.Integral() > 0:
    scale_factor = hist_splot_sig.Integral() / hist_pt_mc.Integral()
    hist_pt_mc.Scale(scale_factor)

# Ajustar eixo Y para caber tudo (pega o máximo entre os dois)
max_y = max(hist_splot_sig.GetMaximum(), hist_pt_mc.GetMaximum())
hist_pt_mc.SetMaximum(max_y * 1.3) # +30% de margem no topo

# Desenhar
hist_pt_mc.Draw("HIST")       # Desenha MC como histograma
hist_splot_sig.Draw("E1 SAME") # Desenha Dados como pontos com erro

# Legenda
legend = ROOT.TLegend(0.6, 0.7, 0.88, 0.88)
legend.AddEntry(hist_splot_sig, "Dados (Sinal via sPlot)", "lep")
legend.AddEntry(hist_pt_mc, "Monte Carlo (Sinal)", "f")
legend.Draw()
c_comp.SetLogy()
c_comp.SaveAs("resultado_final_comparacao.png")

print(">>> Processo finalizado com sucesso!")

>>> Lendo arquivo de DADOS...
   -> Eventos selecionados nos Dados: 818576
>>> Lendo arquivo de MC...
   -> Eventos selecionados no MC: 32989
>>> Construindo modelo de Fit...
>>> Rodando fit...
[#1] INFO:Fitting -- RooAbsPdf::fitTo(model) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using CPU computation library compiled with -mavx2
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_data) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization
[#1] INFO:Plotting -- RooAbsPdf::plotOn(model) directly selected PDF components: (cb_pdf)
[#1] INFO:Plotting -- RooAbsPdf::plotOn(model) indirectly selected PDF components: ()
>>> Calculando sWeights...
[#1] INFO:Plotting -- RooAbsPdf::plotOn(model) directly selected PDF components: (exp_pdf)
[#1] 

Info in <TCanvas::Print>: png file check_fit_mass.png has been created


[#1] INFO:Fitting -- RooAbsPdf::fitTo(model) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_data) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Fitting -- RooAbsPdf::fitTo(model) Calculating sum-of-weights-squared correction matrix for covariance matrix
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization
[#1] INFO:InputArguments -- yield in pdf: n_sig 79589.6
[#1] INFO:InputArguments -- yield in pdf: n_bkg 738993
[#1] INFO:Eval -- Calculating sWeight
>>> Extraindo histogramas ponderados...
>>> Gerando plot de comparação final...
>>> Processo finalizado com sucesso!


Info in <TCanvas::Print>: png file resultado_final_comparacao.png has been created


In [2]:
import ROOT
import math

# Configuração para não mostrar janelas pop-up
ROOT.gROOT.SetBatch(True)

# ==============================================================================
# 1. CONFIGURAÇÃO DAS VARIÁVEIS
# ==============================================================================
# Aqui definimos todas as variáveis extras que queremos plotar via sPlot
features_config = [
    {
        "name": "pt", 
        "branch": "BToTrkTrkMuMu_fit_pt_good", 
        "title": "p_{T} [GeV]", 
        "bins": 50, "min": 0, "max": 50
    },
    {
        "name": "eta", 
        "branch": "BToTrkTrkMuMu_fit_eta_good", 
        "title": "#eta", 
        "bins": 50, "min": -2.5, "max": 2.5
    },
    {
        "name": "phi", 
        "branch": "BToTrkTrkMuMu_fit_phi_good", 
        "title": "#phi", 
        "bins": 50, "min": -3.15, "max": 3.15
    },
    {
        "name": "svprob",
        "branch": "BToTrkTrkMuMu_svprob_good",
        "title": "svprob",
        "bins": 100, "min": 0, "max":1
    }
]

# Variável Discriminante (Massa) - Usada para o Fit
mass = ROOT.RooRealVar("mass", "Massa do B [GeV]", 5.0, 5.6)

# Criar RooRealVars para as variáveis de controle (pT, Eta, Phi)
roo_vars = {}
observables = ROOT.RooArgSet(mass)

# Dicionários para guardar os histogramas
hists_data_splot = {} # Vai receber os dados ponderados
hists_mc = {}         # Vai receber o MC puro

print(">>> Inicializando variáveis e histogramas...")

for feat in features_config:
    # 1. Criar RooRealVar
    roo_var = ROOT.RooRealVar(feat["name"], feat["title"], feat["min"], feat["max"])
    roo_vars[feat["name"]] = roo_var
    observables.add(roo_var)
    
    # 2. Criar Histogramas Vazios (MC e sPlot)
    h_mc = ROOT.TH1F(f"hist_{feat['name']}_mc", f"MC - {feat['title']}", feat["bins"], feat["min"], feat["max"])
    h_splot = ROOT.TH1F(f"hist_{feat['name']}_splot", f"Dados (sPlot) - {feat['title']}", feat["bins"], feat["min"], feat["max"])
    
    h_mc.SetDirectory(0)
    h_splot.SetDirectory(0)
    h_mc.Sumw2()
    h_splot.Sumw2()
    
    hists_mc[feat["name"]] = h_mc
    hists_data_splot[feat["name"]] = h_splot

# Dataset do RooFit
data_roofit = ROOT.RooDataSet("data", "Dataset para Fit", observables)

# ==============================================================================
# 2. FUNÇÃO DE LEITURA (DATA e MC)
# ==============================================================================
def process_file(filepath, is_mc):
    tfile = ROOT.TFile(filepath)
    tree = tfile.Get("tree_ML")
    
    # Otimização de Branches
    tree.SetBranchStatus("*", 0)
    tree.SetBranchStatus("BToTrkTrkMuMu_fit_mass_Kpi_good", 1)
    tree.SetBranchStatus("MuMu_mass_idx_good", 1)
    
    # Ativa branches das variáveis configuradas (pt, eta, phi)
    for feat in features_config:
        tree.SetBranchStatus(feat["branch"], 1)
        
    print(f">>> Lendo {'MC' if is_mc else 'DADOS'}: {filepath}")
    
    counter = 0
    for event in tree:
        masses = event.BToTrkTrkMuMu_fit_mass_Kpi_good
        mumu_masses = event.MuMu_mass_idx_good
        
        if len(masses) == 0: continue
        
        # Leitura das variáveis extras (vetores)
        extra_values = {feat["name"]: getattr(event, feat["branch"]) for feat in features_config}
        
        # Loop sobre candidatos
        for i in range(len(masses)):
            mass_val = masses[i]
            mumu_val = mumu_masses[i]
            
            # CORTES
            if not (2.9 < mumu_val < 3.3): continue # J/Psi
            if not (5.0 < mass_val < 5.6): continue # B Mass
            
            # Se for MC, preenche histogramas direto
            if is_mc:
                for feat in features_config:
                    val = extra_values[feat["name"]][i]
                    hists_mc[feat["name"]].Fill(val)
            
            # Se for DADOS, preenche RooDataSet para o Fit
            else:
                mass.setVal(mass_val)
                # Seta valores das variaveis extras no RooArgSet
                for feat in features_config:
                    val = extra_values[feat["name"]][i]
                    roo_vars[feat["name"]].setVal(val)
                
                data_roofit.add(observables)
                
            counter += 1
            
    print(f"   -> Selecionados: {counter}")
    tfile.Close()

# Executa leitura
process_file("/home/tandrade/Flavour_Anomalies/Data/Data.root", is_mc=False)
process_file("/home/tandrade/Flavour_Anomalies/Data/MC.root", is_mc=True)

# ==============================================================================
# 3. FIT E SPLOT
# ==============================================================================
print(">>> Construindo modelo e realizando Fit...")

# Modelo (Sinal: Crystal Ball, Fundo: Exponencial)
mean  = ROOT.RooRealVar("mean", "Média", 5.28, 5.26, 5.30)
sigma = ROOT.RooRealVar("sigma", "Largura", 0.025, 0.01, 0.05)
alpha = ROOT.RooRealVar("alpha", "Alpha", 1.5, 0.5, 3.0)
n     = ROOT.RooRealVar("n", "n", 5.0, 1.0, 15.0)
cb_pdf = ROOT.RooCBShape("cb_pdf", "Sinal", mass, mean, sigma, alpha, n)

tau = ROOT.RooRealVar("tau", "Decaimento Exp", -2.0, -10.0, 0.0)
exp_pdf = ROOT.RooExponential("exp_pdf", "Fundo", mass, tau)

n_entries = data_roofit.numEntries()
nsig = ROOT.RooRealVar("n_sig", "N Sinal", n_entries * 0.4, 0, n_entries)
nbkg = ROOT.RooRealVar("n_bkg", "N Fundo", n_entries * 0.6, 0, n_entries)

model = ROOT.RooAddPdf("model", "Modelo Total", 
                       ROOT.RooArgList(cb_pdf, exp_pdf), 
                       ROOT.RooArgList(nsig, nbkg))

# Fit
model.fitTo(data_roofit, ROOT.RooFit.Save(True), ROOT.RooFit.PrintLevel(-1), ROOT.RooFit.Extended(True))

# Salva plot de massa para controle
c_mass = ROOT.TCanvas("c_mass", "Fit Massa", 800, 600)
fr = mass.frame()
data_roofit.plotOn(fr)
model.plotOn(fr)
fr.Draw()
c_mass.SaveAs("fit_mass_check.png")

# 

print(">>> Calculando sWeights...")
# Fixar parametros para sPlot
for par in [mean, sigma, alpha, n, tau]: par.setConstant(True)

splot = ROOT.RooStats.SPlot("sData", "sPlot", data_roofit, model, ROOT.RooArgList(nsig, nbkg))

# ==============================================================================
# 4. PROJEÇÃO DOS DADOS PONDERADOS (UNFOLDING)
# ==============================================================================
print(">>> Projetando histogramas ponderados...")

# Iterar sobre o dataset para pegar os pesos e preencher histogramas de Pt, Eta, Phi
for i in range(data_roofit.numEntries()):
    row = data_roofit.get(i)
    weight_sig = row.getRealValue("n_sig_sw") # Peso sPlot de Sinal
    
    for feat in features_config:
        val = row.getRealValue(feat["name"])
        hists_data_splot[feat["name"]].Fill(val, weight_sig)

# ==============================================================================
# 5. PLOTAGEM FINAL
# ==============================================================================
def plot_comparison(feat_name, h_data, h_mc, title):
    c = ROOT.TCanvas(f"c_{feat_name}", title, 800, 600)
    
    # Estilo
    h_data.SetLineColor(ROOT.kBlack)
    h_data.SetMarkerStyle(20)
    h_data.SetMarkerSize(0.8)
    
    h_mc.SetLineColor(ROOT.kRed)
    h_mc.SetLineWidth(2)
    h_mc.SetFillColorAlpha(ROOT.kRed, 0.3)
    
    # Normalização
    if h_mc.Integral() > 0:
        h_mc.Scale(h_data.Integral() / h_mc.Integral())
        
    # Ajuste de eixos
    max_y = max(h_data.GetMaximum(), h_mc.GetMaximum()) * 1.3
    h_mc.SetMaximum(max_y)
    h_mc.SetMinimum(0)
    h_mc.SetTitle(f"Comparacao sPlot vs MC;{title};Eventos")
    
    h_mc.Draw("HIST")
    h_data.Draw("E1 SAME")
    
    leg = ROOT.TLegend(0.65, 0.75, 0.88, 0.88)
    leg.AddEntry(h_data, "Dados (sPlot)", "lep")
    leg.AddEntry(h_mc, "MC (Sinal)", "f")
    leg.Draw()
    
    c.SaveAs(f"sPlot_Comp_{feat_name}.png")

print(">>> Salvando imagens...")
for feat in features_config:
    plot_comparison(feat["name"], 
                   hists_data_splot[feat["name"]], 
                   hists_mc[feat["name"]], 
                   feat["title"])

print(">>> Concluído!")

>>> Inicializando variáveis e histogramas...


OSError: Failed to open file /home/tandrade/Flavour_Anomalies/Data/Data.root

Error in <TFile::TFile>: file /home/tandrade/Flavour_Anomalies/Data/Data.root does not exist
